# 02 · Entrenamiento LoRA (dos configuraciones)

Entrena dos adapters que difieren en **un solo factor**: el rank `r`.

| Config | r | alpha | Hipótesis |
|---|---|---|---|
| A | 8 | 16 | Rank bajo basta para la mayoría de consultas y entrena más rápido. |
| B | 32 | 64 | Más capacidad mejora las consultas difíciles (subconsultas, ventanas). |

Todo lo demás se mantiene igual (lr=2e-4, 100 pasos con batch efectivo 16, mismos módulos, LoRA fp16).

In [ ]:
# Clonar el repositorio en Colab (ajusta la URL a tu repo privado)
# !git clone https://github.com/USUARIO/proyecto-dl-3.git
# %cd proyecto-dl-3
import sys, os
# Si ejecutas localmente, asegúrate de estar en la raíz del repo.
assert os.path.exists("src"), "Ejecuta desde la raíz del repositorio."


In [ ]:
!pip -q install -r requirements.txt


## Entrenar config A (r=8)

In [ ]:
from src.entrenar import entrenar
# LoRA fp16 (bitsandbytes no cargó en el CUDA de Colab); batch efectivo 16.
rep_A = entrenar(config='A', salida='adapter_A', epocas=1, max_pasos=100,
                usar_qlora=False, batch=2, grad_accum=8)
rep_A


## Entrenar config B (r=32)

In [ ]:
rep_B = entrenar(config='B', salida='adapter_B', epocas=1, max_pasos=100,
                usar_qlora=False, batch=2, grad_accum=8)
rep_B


## Curvas de loss

In [ ]:
import matplotlib.pyplot as plt, json
for nombre in ['adapter_A','adapter_B']:
    r=json.load(open(f'{nombre}/reporte_entrenamiento.json'))
    pasos=[h['step'] for h in r['loss_history']]
    loss=[h['loss'] for h in r['loss_history']]
    plt.plot(pasos, loss, label=f"{nombre} (r={r['r']})")
plt.xlabel('paso'); plt.ylabel('loss'); plt.legend(); plt.title('Curvas de loss')
plt.savefig('resultados/curvas_loss.png', dpi=120, bbox_inches='tight'); plt.show()


## Elegir el mejor adapter
Tras evaluar (notebook 03), copia el mejor a `adapter/` para la entrega:
`!rm -rf adapter && cp -r adapter_B adapter`  (o adapter_A según EX).